In [ ]:
#@title 1. Configuración del Entorno
import os

# Clonar el repositorio si no existe
if not os.path.exists("Signal-Analysis-Vol-I"):
    !git clone https://github.com/ecundir/Signal-Analysis-Vol-I.git

# Cambiar a la carpeta del capítulo
%cd Signal-Analysis-Vol-I/capitulo_7
print("✅ Entorno configurado correctamente.")

**ESTABILIDAD EN EL PLANO s**

Una vez desplegada la interfaz, utiliza el deslizador de Sigma (σ) para desplazar los polos horizontalmente; notarás que mientras los polos se mantengan en el semiplano izquierdo (valores negativos), el sistema será estable y la región de convergencia cubrirá el eje vertical. Al mover Omega (ω), estarás ajustando la frecuencia de oscilación natural del sistema, lo cual se verá reflejado inmediatamente en la posición vertical de las "X" rojas en el plano complejo y en la rapidez de los ciclos en la gráfica de respuesta al impulso.

Observa la transición crítica de estabilidad. Te sugiero llevar el control de Sigma hacia valores positivos para forzar el cruce de los polos al semiplano derecho; verás cómo el indicador cambia de "ESTABLE" a "INESTABLE" y la señal temporal comienza a crecer de forma exponencial, simulando un sistema que ha perdido el control. Este ejercicio práctico te permitirá visualizar cómo la posición de las raíces en el plano complejo dicta el destino físico de cualquier equipo de telecomunicaciones, transformando un filtro funcional en un oscilador peligroso con solo deslizar un parámetro.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import interact, FloatSlider

def lab_vivo_laplace(sigma=-2.0, omega=10.0):
    # s = sigma +/- j*omega
    polos = [complex(sigma, omega), complex(sigma, -omega)]
    sys = signal.lti([], polos, 1)

    t = np.linspace(0, 5, 1000)
    w = np.linspace(0, 150, 1000)

    t_imp, y_imp = signal.impulse(sys, T=t)
    w_freq, mag, _ = signal.bode(sys, w=w)

    fig = plt.figure(figsize=(15, 10))

    # --- Plano S ---
    ax1 = fig.add_subplot(2, 2, 1)
    ax1.axvline(0, color='black', lw=1)
    ax1.axhline(0, color='black', lw=1)
    ax1.axvspan(sigma, 10, color='green', alpha=0.1, label='ROC (Causal)')
    ax1.scatter([sigma, sigma], [omega, -omega], marker='x', color='red', s=100)

    # SOLUCIÓN: Usar r"..." para evitar SyntaxWarning
    ax1.set_title(r"Plano Complejo $s$ (El Mapa)")
    ax1.set_xlabel(r"$\sigma$ (Amortiguamiento)")
    ax1.set_ylabel(r"$j\omega$ (Oscilación)")
    ax1.set_xlim([-10, 10])
    ax1.set_ylim([-50, 50])
    ax1.grid(True, alpha=0.3)

    # --- Respuesta al Impulso ---
    ax2 = fig.add_subplot(2, 2, 2)
    ax2.plot(t_imp, y_imp, color='blue', lw=2)
    ax2.set_title(r"Respuesta al Impulso $h(t)$")
    ax2.grid(True, alpha=0.3)

    status = "ESTABLE" if sigma < 0 else "INESTABLE"
    color_st = "darkgreen" if sigma < 0 else "red"
    ax2.text(0.6, 0.85, status, transform=ax2.transAxes, color=color_st,
             fontsize=14, fontweight='bold', bbox=dict(facecolor='white', alpha=0.8))

    # --- Magnitud ---
    ax3 = fig.add_subplot(2, 1, 2)
    ax3.semilogx(w_freq, mag, color='purple', lw=2)
    # SOLUCIÓN: r"..." aquí también
    ax3.set_title(r"Respuesta en Magnitud $|H(j\omega)|$ (Corte en el eje $j\omega$)")
    ax3.set_xlabel("Frecuencia (rad/s)")
    ax3.set_ylabel("Ganancia (dB)")
    ax3.grid(True, which="both", alpha=0.3)

    plt.tight_layout()
    plt.show()

interact(lab_vivo_laplace,
         sigma=FloatSlider(min=-10.0, max=2.0, step=0.2, value=-2.0, description='Sigma (σ)'),
         omega=FloatSlider(min=0.0, max=40.0, step=1.0, value=10.0, description='Omega (ω)'));